# yt-clips — Colab Worker

### Before this session (on your Mac):
```bash
python push_code.py --config-only    # sync secrets (.env, tokens) to Drive
```

### On Colab:
1. **Runtime → Change runtime type → T4 GPU**
2. Run **Cell 1** — mount Drive + git clone code + load secrets
3. Run **Cell 2** — install deps
4. Run **Cell 3** — start watcher + tunnel
5. Run **Cell 4** — monitor (keep running)

### Submit jobs (on your Mac):
```bash
python bridge.py "https://youtu.be/VIDEO_ID"
# If no tunnel:  python bridge.py saves locally; manually inject into Colab
```

**Code comes from Git** — secrets come from Drive.

In [ ]:
# Cell 1 — Mount Drive → git clone → load secrets
from google.colab import drive
import os, sys, shutil, subprocess
from pathlib import Path

drive.mount("/content/drive")

DRIVE_DIR = None
for p in ["/content/drive/MyDrive/yt-clips", "/content/drive/My Drive/yt-clips"]:
    if Path(p).exists():
        DRIVE_DIR = p
        break

if not DRIVE_DIR:
    raise SystemExit("yt-clips folder not found on Drive! Run python push_code.py --config-only on Mac first.")

print(f"Secrets dir: {DRIVE_DIR}")

# ── Git clone / pull code ──
REPO = "/content/yt-clips"
if Path(REPO).exists():
    subprocess.run(["git", "-C", REPO, "pull"], capture_output=True)
    print("Code: git pull")
else:
    subprocess.run(["git", "clone", "https://github.com/fearless16/yt-clips", REPO], check=True)
    print("Code: git clone")

os.chdir(REPO)
sys.path.insert(0, REPO)
print(f"Working dir: {REPO}")

# ── Copy secrets from Drive ──
SECRET_FILES = [".env", "yt_token.json", "yt_analytics_token.json",
               "client_secrets.json", "cookies.txt",
               "channel_logo.png", "face_landmarker.task",
               "remote_job.json", "colab_url.txt"]
for fname in SECRET_FILES:
    src = Path(DRIVE_DIR) / fname
    dst = Path(REPO) / fname
    if src.exists():
        shutil.copy2(src, dst)
        print(f"  Secrets: {fname}")

# ── Load .env ──
env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and "=" in line and not line.startswith("#"):
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip()
    print(f".env loaded ({env_path.stat().st_size} bytes)")
else:
    print("WARNING: no .env on Drive — API keys may be missing")

In [ ]:
# Cell 2 — Install system + Python deps
import subprocess, sys, os, time
from pathlib import Path

def sh(cmd, timeout=120):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if r.returncode and r.stderr:
        print(r.stderr.strip()[-200:])
    return r

print("Installing system deps...")
sh("apt-get update -qq && apt-get install -y -qq aria2 ffmpeg >/dev/null 2>&1", 120)

print("Installing deno (for yt-dlp JS solver)...")
sh("curl -fsSL https://deno.land/x/install/install.sh | sh > /dev/null 2>&1", 60)
deno_bin = os.path.expanduser("~/.deno/bin")
os.environ["PATH"] = deno_bin + ":" + os.environ.get("PATH", "")
print(f"  deno: {sh('deno --version 2>&1 | head -1').stdout.strip() or 'FAILED'}")

print("Installing Python deps...")
sh("pip install -q yt-dlp faster-whisper youtube-transcript-api "
   "rich PyYAML opencv-python-headless numpy "
   "filterpy scipy openai python-dotenv Pillow requests "
   "ultralytics gfpgan basicsr realesrgan "
   "google-api-python-client google-auth-httplib2 google-auth-oauthlib", 300)

print("Installing ngrok...")
sh("pip install -q pyngrok", 60)

print("Installing PyTorch (CUDA 12.1)...")
sh("pip install -q torch torchvision torchaudio "
   "--extra-index-url https://download.pytorch.org/whl/cu121", 300)

import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import automation
print(f"automation v{automation.VERSION}")

for d in ["input", "temp", "transcripts", "highlights", "shorts", "logs", "photos"]:
    Path(d).mkdir(exist_ok=True)
print("Directories ready")

In [ ]:
# Cell 3 — Start watcher + tunnel
# Watcher processes jobs submitted via tunnel (POST /job).
# Tunnel is OPTIONAL — job delivery is tunnel-only, no Drive polling.
import subprocess, sys, time, os, re, threading
from pathlib import Path

def _extract_url(log_path, patterns, exclude=()):
    if not log_path.exists():
        return None
    for line in log_path.read_text().splitlines():
        for p in patterns:
            m = re.search(p, line)
            if m:
                url = m.group(1)
                if not any(ex in url for ex in exclude):
                    return url
    return None

def _show_log_tail(log_path, n=10):
    if log_path.exists():
        for line in log_path.read_text().splitlines()[-n:]:
            print(f"  | {line}")

def _drive_sync_loop(drive_dir, repo_dir, interval=5):
    """Background thread: sync tunnel URL from Drive to repo. Drive is credentials-only."""
    while True:
        time.sleep(interval)
        for fname in ["colab_url.txt"]:
            src = Path(drive_dir) / fname
            dst = Path(repo_dir) / fname
            if src.exists() and (not dst.exists() or src.stat().st_mtime != dst.stat().st_mtime):
                shutil.copy2(src, dst)
                print(f"[DRIVE] Synced {fname}")

# ── Kill any stale processes ──
subprocess.run("pkill -9 -f 'watcher.py' 2>/dev/null; sleep 1", shell=True)
subprocess.run("pkill -f ngrok 2>/dev/null; pkill -f serveo 2>/dev/null", shell=True)
subprocess.run("fuser -k 5000/tcp 2>/dev/null; sleep 1", shell=True)
time.sleep(2)

# ── Start watcher (background) ──
watcher_log = open("watcher.log", "w")
watcher_proc = subprocess.Popen(
    [sys.executable, "watcher.py"],
    stdout=watcher_log, stderr=subprocess.STDOUT
)
time.sleep(3)

# Verify watcher is up
import urllib.request
try:
    r = urllib.request.urlopen("http://localhost:5000/health", timeout=3)
    print(f"Watcher: OK (PID {watcher_proc.pid})")
    # ── Start Drive sync thread ──
    sync_thread = threading.Thread(
        target=_drive_sync_loop,
        args=(DRIVE_DIR, os.getcwd(), 5),
        daemon=True,
    )
    sync_thread.start()
    print("  Drive sync thread started (polls remote_job.json every 5s)")
except Exception:
    print("Watcher: FAILED — check watcher.log below")
    _show_log_tail(Path("watcher.log"))

# ── Start tunnel ──
tunnel_url = None
NGROK_DOMAIN = "wiry-rubble-boring.ngrok-free.dev"

# Try fixed ngrok domain
print(f"Starting tunnel (ngrok {NGROK_DOMAIN})...")
try:
    from pyngrok import ngrok
    tunnel_url = ngrok.connect(5000, domain=NGROK_DOMAIN).public_url
    print(f"Tunnel URL: {tunnel_url}")
except Exception as e:
    print(f"ngrok failed: {e}")

# Fallback: serveo
if not tunnel_url:
    tunnel_log = open("tunnel.log", "w")
    print("Trying serveo...")
    sv_proc = subprocess.Popen(
        ["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30",
         "-R", "80:localhost:5000", "serveo.net"],
        stdout=tunnel_log, stderr=subprocess.STDOUT
    )
    time.sleep(8)
    tunnel_url = _extract_url(
        Path("tunnel.log"),
        [r"(https://(?:[a-zA-Z0-9-]+\.)?[a-zA-Z0-9-]+\.serveousercontent\.com)",
         r"(https://[a-zA-Z0-9-]+\.serveo\.net)"],
        exclude=["console"],
    )
    if not tunnel_url:
        print("serveo failed:")
        _show_log_tail(Path("tunnel.log"))

if tunnel_url:
    Path("colab_url.txt").write_text(tunnel_url)
    print(f"Tunnel URL: {tunnel_url}")
    print("Mac → python bridge.py \"URL\"")
else:
    print("⚠ No tunnel available. Check serveo/localtunnel above.")
    print("  Save colab_url.txt to Drive via the Drive sync for Mac to pick up.")

print("\nWatcher is running. Continue to Cell 4 to monitor.")

In [ ]:
# Cell 4 — Monitor (keep this cell running)
# Stop with ■ (interrupt kernel) when done.
import time, json
from pathlib import Path

print("=" * 55)
print("  WORKER IS ONLINE")
print("=" * 55)
tunnel_url = Path("colab_url.txt").read_text().strip() if Path("colab_url.txt").exists() else None
if tunnel_url:
    print(f"  Tunnel: {tunnel_url}")
else:
    print("  No tunnel — check Cell 3")
print("=" * 55)

log_pos = 0
try:
    while True:
        time.sleep(10)

        # ── Tail watcher.log ──
        log_path = Path("watcher.log")
        if log_path.exists():
            with open(log_path) as f:
                if log_path.stat().st_size < log_pos:
                    log_pos = 0  # log was rotated/truncated
                f.seek(log_pos)
                for line in f:
                    if line.strip():
                        print(line.strip())
                log_pos = f.tell()

        # ── Show status.json if it exists ──
        status_path = Path("status.json")
        if status_path.exists():
            try:
                s = json.loads(status_path.read_text())
                print(f"[STATUS] {s.get('status','?')} — {s.get('message','')}")
            except Exception:
                pass

        # ── Show GPU/RAM once per minute ──
        if int(time.time()) % 60 < 12:
            import torch
            has_gpu = torch.cuda.is_available()
            if has_gpu:
                free, total = torch.cuda.mem_get_info()
                print(f"[GPU] {torch.cuda.get_device_name(0)}  VRAM: {(total-free)/1e9:.1f}/{total/1e9:.1f} GB")
            mem = Path("/proc/meminfo")
            if mem.exists():
                free = int(open(mem).read().split("MemFree:")[1].split()[0]) / 1e6
                total = int(open(mem).read().split("MemTotal:")[1].split()[0]) / 1e6
                print(f"[RAM] {total-free:.1f}/{total:.1f} GB used  ({free:.1f} GB free)")

except KeyboardInterrupt:
    print("\nMonitor stopped.")